# 03 特征工程

In [ ]:
TOPIC_WEIBO_PATH = r'..\data\crawler\topic_weibo.parquet'
TOPIC_COMMENT_PATH = r'..\data\crawler\topic_comment.parquet'
USER_INFO_PATH = r'..\data\crawler\user_info.parquet'
USER_WEIBO_PATH = r'..\data\crawler\user_weibo.parquet'

In [ ]:
import pandas as pd

df_topic_weibo = pd.read_parquet(TOPIC_WEIBO_PATH)
df_topic_comment = pd.read_parquet(TOPIC_COMMENT_PATH)
df_user_info = pd.read_parquet(USER_INFO_PATH)
df_user_weibo = pd.read_parquet(USER_WEIBO_PATH)

## 📊 数据探索

先全面了解四个数据表的结构、字段类型、缺失情况和基本统计特征，为后续清洗策略提供依据。

In [ ]:
print("=" * 60)
print("📋 df_topic_weibo（话题微博）")
print(f"Shape: {df_topic_weibo.shape}")
print(f"\nColumns: {df_topic_weibo.columns.tolist()}")
print(f"\nDtypes:\n{df_topic_weibo.dtypes}")
print(f"\n缺失值:\n{df_topic_weibo.isnull().sum()}")
print(f"\n前3行:")
df_topic_weibo.head(3)

In [ ]:
print("=" * 60)
print("📋 df_topic_comment（话题评论）")
print(f"Shape: {df_topic_comment.shape}")
print(f"\nColumns: {df_topic_comment.columns.tolist()}")
print(f"\nDtypes:\n{df_topic_comment.dtypes}")
print(f"\n缺失值:\n{df_topic_comment.isnull().sum()}")
print(f"\n前3行:")
df_topic_comment.head(3)

In [ ]:
print("=" * 60)
print("📋 df_user_info（用户信息）")
print(f"Shape: {df_user_info.shape}")
print(f"\nColumns: {df_user_info.columns.tolist()}")
print(f"\nDtypes:\n{df_user_info.dtypes}")
print(f"\n缺失值:\n{df_user_info.isnull().sum()}")
print(f"\n前3行:")
df_user_info.head(3)

In [ ]:
print("=" * 60)
print("📋 df_user_weibo（用户历史微博）")
print(f"Shape: {df_user_weibo.shape}")
print(f"\nColumns: {df_user_weibo.columns.tolist()}")
print(f"\nDtypes:\n{df_user_weibo.dtypes}")
print(f"\n缺失值:\n{df_user_weibo.isnull().sum()}")
print(f"\n前3行:")
df_user_weibo.head(3)

In [ ]:
# 深入探索文本和行为特征
print("=" * 60)
print("🔍 深入数据特征分析")
print("=" * 60)

# 1. 各表重复记录
print("\n【重复记录检查】")
print(f"  topic_weibo  重复 weibo_id: {df_topic_weibo['weibo_id'].duplicated().sum()}")
print(f"  topic_comment 重复 comment_id: {df_topic_comment['comment_id'].duplicated().sum()}")
print(f"  user_info     重复 user_id: {df_user_info['user_id'].duplicated().sum()}")
print(f"  user_weibo    重复 weibo_id: {df_user_weibo['weibo_id'].duplicated().sum()}")

# 2. 文本长度分布
print("\n【文本长度统计 (字符数)】")
for name, df, col in [
    ("topic_weibo", df_topic_weibo, "content"),
    ("topic_comment", df_topic_comment, "content"),
    ("user_weibo", df_user_weibo, "content"),
]:
    lengths = df[col].str.len()
    print(f"  {name}: mean={lengths.mean():.1f}, median={lengths.median():.1f}, "
          f"min={lengths.min()}, max={lengths.max()}")

# 3. 空或极短文本
print("\n【极短文本 (<=2字符)】")
for name, df, col in [
    ("topic_weibo", df_topic_weibo, "content"),
    ("topic_comment", df_topic_comment, "content"),
    ("user_weibo", df_user_weibo, "content"),
]:
    short_count = (df[col].str.len() <= 2).sum()
    print(f"  {name}: {short_count} 条 ({short_count/len(df)*100:.2f}%)")

# 4. 转发微博特征（user_weibo）
print("\n【user_weibo 转发分析】")
repost_mask = df_user_weibo["reposted_weibo_id"] != -1
print(f"  转发微博: {repost_mask.sum()} 条 ({repost_mask.sum()/len(df_user_weibo)*100:.1f}%)")
print(f"  原创微博: {(~repost_mask).sum()} 条 ({(~repost_mask).sum()/len(df_user_weibo)*100:.1f}%)")

# 5. "转发微博" 纯文本
pure_repost = (df_user_weibo["content"] == "转发微博").sum()
print(f"  内容为'转发微博': {pure_repost} 条 ({pure_repost/len(df_user_weibo)*100:.1f}%)")

# 6. 话题分布
print("\n【topic_weibo 话题分布】")
print(df_topic_weibo["topic"].value_counts().head(10))

# 7. 性别分布
print("\n【性别分布】")
for name, df in [("topic_weibo", df_topic_weibo), ("topic_comment", df_topic_comment),
                  ("user_info", df_user_info)]:
    print(f"  {name}: {df['gender'].value_counts().to_dict()}")

# 8. user_info 中 user_id 类型
print(f"\n【user_info.user_id 类型: {df_user_info['user_id'].dtype}】")
print(f"  样例: {df_user_info['user_id'].head(3).tolist()}")

In [ ]:
import re

# 微博文本中的特殊内容特征分析
print("=" * 60)
print("🔍 文本内容特征分析（微博特有元素）")
print("=" * 60)

sample_texts = df_topic_weibo["content"].head(20).tolist() + df_user_weibo["content"].head(20).tolist()

# 检测常见模式
patterns = {
    "URL链接": r'https?://\S+',
    "话题标签 #xxx#": r'#[^#]+#',
    "@用户": r'@[\w\u4e00-\u9fff]+',
    "表情 [xxx]": r'\[[^\[\]]+\]',
    "HTML标签": r'<[^>]+>',
    "转发标记": r'转发微博',
    "回复标记": r'回复@',
}

for table_name, df in [("topic_weibo", df_topic_weibo), ("topic_comment", df_topic_comment),
                        ("user_weibo", df_user_weibo)]:
    print(f"\n--- {table_name} (共{len(df)}条) ---")
    for pat_name, pat in patterns.items():
        count = df["content"].str.contains(pat, regex=True, na=False).sum()
        print(f"  {pat_name}: {count} 条 ({count/len(df)*100:.1f}%)")

# 查看 topic_comment 中空文本情况
print("\n--- topic_comment 空文本内容 ---")
empty_comments = df_topic_comment[df_topic_comment["content"].str.len() == 0]
print(f"完全空文本: {len(empty_comments)} 条")

# user_weibo 中内容为空的情况
print("\n--- user_weibo 空文本内容 ---")
empty_weibo = df_user_weibo[df_user_weibo["content"].str.len() == 0]
print(f"完全空文本: {len(empty_weibo)} 条")

# 查看 topic_comment 中极短文本样例
print("\n--- topic_comment 极短文本样例 (<=2字符) ---")
short_comments = df_topic_comment[df_topic_comment["content"].str.len() <= 2]["content"].value_counts().head(15)
print(short_comments)

In [ ]:
# 重复 weibo_id 分析 & 数据关系完整性
print("=" * 60)
print("🔍 user_weibo 重复分析 & 表间关系")
print("=" * 60)

# user_weibo 重复 weibo_id 是什么情况？
dup_ids = df_user_weibo[df_user_weibo["weibo_id"].duplicated(keep=False)]
print(f"\n重复 weibo_id 涉及 {dup_ids['weibo_id'].nunique()} 个唯一 weibo_id，共 {len(dup_ids)} 行")
# 是同一用户的重复还是不同用户的？
dup_sample = dup_ids.groupby("weibo_id").agg(
    n_users=("user_id", "nunique"),
    n_rows=("user_id", "count")
).reset_index()
print(f"多用户重复（转发同一条）: {(dup_sample['n_users'] > 1).sum()}")
print(f"同用户重复（完全重复）: {(dup_sample['n_users'] == 1).sum()}")

# 表间关系
print("\n--- 表间关联分析 ---")
# topic_comment 的 weibo_id 是否都在 topic_weibo 中
comment_weibo_ids = set(df_topic_comment["weibo_id"].unique())
topic_weibo_ids = set(df_topic_weibo["weibo_id"].unique())
print(f"topic_comment 中 weibo_id 能关联到 topic_weibo: "
      f"{len(comment_weibo_ids & topic_weibo_ids)}/{len(comment_weibo_ids)}")

# user_info 与 topic_weibo/topic_comment 的用户覆盖
user_info_ids = set(df_user_info["user_id"].astype(str).unique())
topic_weibo_user_ids = set(df_topic_weibo["user_id"].astype(str).unique())
topic_comment_user_ids = set(df_topic_comment["user_id"].astype(str).unique())
user_weibo_user_ids = set(df_user_weibo["user_id"].astype(str).unique())

print(f"user_info 覆盖 topic_weibo 用户: "
      f"{len(user_info_ids & topic_weibo_user_ids)}/{len(topic_weibo_user_ids)}")
print(f"user_info 覆盖 topic_comment 用户: "
      f"{len(user_info_ids & topic_comment_user_ids)}/{len(topic_comment_user_ids)}")
print(f"user_weibo 覆盖 user_info 用户: "
      f"{len(user_weibo_user_ids & user_info_ids)}/{len(user_info_ids)}")

# user_info 中有多少用户在 user_weibo 中有微博？
print(f"\n--- user_weibo 每用户微博数量分布 ---")
print(df_user_weibo.groupby("user_id").size().describe())

## 特征工程

In [ ]:
from pandas import DataFrame

def add_time_features(df: DataFrame, time_col: str = "create_time") -> DataFrame:
    """从时间列提取多维度时间特征。"""
    df["year"] = df[time_col].dt.year
    df["month"] = df[time_col].dt.month
    df["day"] = df[time_col].dt.day
    df["hour"] = df[time_col].dt.hour
    df["weekday"] = df[time_col].dt.day_name()
    return df

### `df_topic_weibo` 表

In [ ]:
# 时间特征
df_topic_weibo = add_time_features(df_topic_weibo)

# 文本长度
df_topic_weibo["text_length"] = df_topic_weibo["content"].str.len()

# 互动量
# topic_weibo: like + comment + repost
df_topic_weibo["engagement"] = (
    df_topic_weibo["like_count"]
    + df_topic_weibo["comment_count"]
    + df_topic_weibo["repost_count"]
)

In [ ]:
# topic_weibo 最终字段
topic_weibo_cols = [
    # ID & 用户
    "weibo_id", "user_id", "screen_name", "gender",
    # 话题
    "topic",
    # 文本
    "content", "text_length",
    # 时间
    "create_time", "year", "month", "day", "hour", "weekday",
    # 互动
    "like_count", "comment_count", "repost_count", "engagement",
    # 爬取评论数
    # "crawled_comment_count",
    # 热搜词条信息
    "trending_date", "trending_type", "trending_click"
]
df_topic_weibo = df_topic_weibo[topic_weibo_cols]

### `df_topic_comment` 表

In [ ]:
# 时间特征
df_topic_comment = add_time_features(df_topic_comment)

# 文本长度
df_topic_comment["text_length"] = df_topic_comment["content"].str.len()

# 互动量
# topic_comment: like + sub_comment_count
df_topic_comment["engagement"] = (
    df_topic_comment["like_count"]
    + df_topic_comment["sub_comment_count"]
)

In [ ]:
# 计算每条评论在数据集中的子评论数量
# parent_id != -1 表示该评论是某个评论的回复（二级评论）
# 统计有多少条评论的 parent_id 等于当前评论的 comment_id
sub_comment_counts = (
    df_topic_comment[df_topic_comment["parent_id"] != -1]
    .groupby("parent_id")
    .size()
    .reset_index(name="sub_comment_crawled_count")
)
sub_comment_counts.rename(columns={"parent_id": "comment_id"}, inplace=True)
df_topic_comment = df_topic_comment.merge(
    sub_comment_counts, on="comment_id", how="left"
)
df_topic_comment["sub_comment_crawled_count"] = (
    df_topic_comment["sub_comment_crawled_count"].fillna(0).astype(int)
)

print(f"✅ topic_comment 添加 sub_comment_crawled_count: "
      f"有子评论 {(df_topic_comment['sub_comment_crawled_count'] > 0).sum()} 条, "
      f"无子评论 {(df_topic_comment['sub_comment_crawled_count'] == 0).sum()} 条")

In [ ]:
# 文本质量等级
df_topic_comment["text_quality"] = 3  # 先设初值为3
df_topic_comment["text_quality_label"] = "可分析"
# 在后续的清洗阶段，根据实际情况调整 text_quality 和 text_quality_label

In [ ]:
# df_topic_comment 最终字段
topic_comment_cols = [
    # ID & 关联
    "comment_id", "weibo_id", "parent_id",
    # 用户
    "user_id", "screen_name", "gender",
    # 文本
    "content", "text_length", "text_quality", "text_quality_label",
    # 时间
    "create_time", "year", "month", "day", "hour", "weekday",
    # 互动
    "like_count", "sub_comment_count", "engagement", 
    "sub_comment_crawled_count",
    # 位置
    "ip_location",
]
df_topic_comment = df_topic_comment[topic_comment_cols]

### `df_user_weibo` 表

In [ ]:
# 时间特征
df_user_weibo = add_time_features(df_user_weibo)

# 文本长度
df_user_weibo["text_length"] = df_user_weibo["content"].str.len()

# 添加转发标记
df_user_weibo["is_repost"] = df_user_weibo["reposted_weibo_id"] != -1
print(f"✅ user_weibo: 转发 {df_user_weibo['is_repost'].sum():,} 条, "
      f"原创 {(df_user_weibo['is_repost'] == 0).sum():,} 条")

# 互动量
# df_user_weibo: like + comment + repost
df_user_weibo["engagement"] = (
    df_user_weibo["like_count"]
    + df_user_weibo["comment_count"]
    + df_user_weibo["repost_count"]
)

In [ ]:
# 活跃度统计
# 计算每位用户在 user_weibo 中的历史微博数量
# user_post_counts = df_user_weibo.groupby("user_id").size().reset_index(name="user_post_count")
# df_user_weibo = df_user_weibo.merge(user_post_counts, on="user_id", how="left")

# 计算每位用户的原创微博比例
user_original_ratio = (
    df_user_weibo.groupby("user_id")["is_repost"]
    .apply(lambda x: 1 - x.mean())
    .reset_index(name="original_ratio")
)
df_user_weibo = df_user_weibo.merge(user_original_ratio, on="user_id", how="left")

In [ ]:
# 文本质量等级
df_user_weibo["text_quality"] = 3  # 先设初值为3
df_user_weibo["text_quality_label"] = "可分析"
# 在后续的清洗阶段，根据实际情况调整 text_quality 和 text_quality_label

In [ ]:
# df_user_weibo 最终字段
user_weibo_cols = [
    # ID & 用户
    "weibo_id", "user_id", "screen_name",
    # 文本
    "content", "text_length",
    # 文本质量
    "text_quality", "text_quality_label",
    # 时间
    "create_time", "year", "month", "day", "hour", "weekday",
    # 互动
    "like_count", "comment_count", "repost_count", "engagement",
    # 转发关系
    "is_repost", "reposted_weibo_id",
    # 社交元数据
    "topics", "at_users",
    # 用户维度统计
    # "user_post_count", "original_ratio",
]
df_user_weibo = df_user_weibo[user_weibo_cols]

### `df_user_info` 表

In [ ]:
# 统一 user_id 类型为 int64
df_user_info["user_id"] = df_user_info["user_id"].astype("int64")

# registration_time 转 datetime
df_user_info["registration_time"] = pd.to_datetime(df_user_info["registration_time"], errors="coerce")

# 计算账号年龄（天数）
reference_date = pd.Timestamp("2026-03-01")
df_user_info["account_age_days"] = (reference_date - df_user_info["registration_time"]).dt.days
df_user_info["account_age_days"] = df_user_info["account_age_days"].fillna(-1).astype(int)

# ip_location 处理
df_user_info["ip_location"] = df_user_info["ip_location"].replace({"未知": None, "": None})

# 粉丝/关注比（影响力指标）
df_user_info["follower_following_ratio"] = (
    df_user_info["follower_count"] / df_user_info["following_count"].replace(0, 1)
).round(2)

In [ ]:
# df_user_info 最终字段
user_info_cols = [
    # ID
    "user_id", "screen_name", "gender",
    # 位置
    "ip_location",
    # 账号信息
    "registration_time", "account_age_days",
    "verified", "verified_type", "verified_type_name",
    # 社交指标
    "total_weibo_count", "follower_count", "following_count",
    "follower_following_ratio", "user_rank",
    # 活跃度
    # "crawled_weibo_count", "avg_engagement", "original_ratio",
    # 个人简介
    "description",
]
df_user_info = df_user_info[user_info_cols]

## 结果保存

In [ ]:
import os

output_dir = r"..\data\fe"

datasets = {
    "topic_weibo": df_topic_weibo,
    "topic_comment": df_topic_comment,
    "user_info": df_user_info,
    "user_weibo": df_user_weibo,
}

for name, df in datasets.items():
    path = os.path.join(output_dir, f"{name}.parquet")
    df.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name}.parquet 已保存 ({df.shape[0]:>10,} rows × {df.shape[1]:>2} cols, {size_mb:.1f} MB)")

print(f"\n📂 输出目录: {os.path.abspath(output_dir)}")
